# レッスン 04 - ツール使用デザインパターン

このレッスンでは、Microsoft Agent Framework (Python) を使った AI エージェント向けの **ツール使用（Tool Use）** デザインパターンを学びます。内容は以下の通りです：

- `@tool` デコレーターと型付きパラメーターで関数ツールを定義する方法
- 各ツールが何をするかをモデルに知らせるためのツールスキーマの提供
- `approval_mode` を用いたツール実行の制御
- Pydantic モデルと `response_format` を通じた<strong>構造化された出力</strong>の返却

シナリオは、目的地を調べ、空き状況を確認し、フライト情報を取得できる<strong>旅行予約エージェント</strong>です。


## セットアップ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool デコレータを使ったツール定義

`@tool` デコレータは、通常のPython関数をエージェントが呼び出せるツールに変えます。
重要なポイント:

- <strong>ドキュメンテーション文字列</strong> はモデルが見るツールの説明になります。
- <strong>型アノテーション</strong>（説明付きの `Annotated` を含む）がツールスキーマを定義します。
- `approval_mode` は実行前にユーザーが各呼び出しを承認する必要があるかどうかを制御します。


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## 複数ツールを持つエージェントの作成

モデルがユーザーの質問に答えるために必要なツールを呼び出せるように、3つすべてのツールをクライアントに渡します。


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## ツールによる構造化出力

`response_format` を Pydantic モデルに設定することで、エージェントは自由形式のテキストではなく、型が明確な JSON オブジェクトを返すことを強制されます。これは、下流のコードが結果をプログラム的に処理する必要がある場合に役立ちます。


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## ツール承認パターン

`@tool` の `approval_mode` パラメーターは、ツールの呼び出しが実行前に人間の承認を必要とするかどうかを制御します：

| モード | 動作 |
|---|---|
| `"never_require"` | ツールが自動的に実行され、ユーザーの確認は不要です。 |
| `"always_require"` | 実行前にすべての呼び出しがユーザーによって承認されなければなりません。 |

副次的な影響のあるツール（例：フライトの予約、クレジットカードの請求）には `"always_require"` を使用して、人間が関与し続けるようにしてください。


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## まとめ

このレッスンでは、以下のことを学びました:

1. 型付きパラメーターとツールスキーマとして機能するドキュメント文字列を持つ `@tool` デコレーターを使って<strong>ツールを定義する</strong>方法。
2. エージェントが複雑なクエリに答えるために順次ツールを呼び出せるように<strong>複数のツールを組み合わせる</strong>方法。
3. Pydanticモデルを `response_format` として渡して<strong>構造化された出力を返す</strong>方法。
4. `approval_mode` を使って、重要な操作に人間の承認を介入させる<strong>ツール承認の制御</strong>方法。

これらのパターンは、外部システムと安全にやり取りできる信頼性が高く本番環境に適したエージェントを構築するための基盤を形成します。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
